In [5]:
!pip install scipy

     ---------------------------------------- 41.3/41.3 MB 7.3 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Shubhangi\\Documents\\GitHub\\MGNREGA-Data-Analysis\\data\\raw\\MERGED_SET.csv")
df.head()

,fin_year,month,state_code,state_name,district_code,district_name,Approved_Labour_Budget,Average_Wage_rate_per_day_per_person,Average_days_of_employment_provided_per_Household,Differently_abled_persons_worked,...,Total_No_of_JobCards_issued,Total_No_of_Workers,Total_No_of_Works_Takenup,Wages,Women_Persondays,percent_of_Category_B_Works,percent_of_Expenditure_on_Agriculture_Allied_Works,percent_of_NRM_Expenditure,percentage_payments_gererated_within_15_days,Remarks
0,2018-2019,Nov,2,ANDHRA PRADESH,202,VIZIANAGARAM,15623529,198.180663,48,10878,...,519227,1218221,130223,34032.16018,10985218,57,76.08,74.11,105.71,NaN
1,2018-2019,Nov,2,ANDHRA PRADESH,209,NELLORE,10931309,201.456988,50,3235,...,615947,1261092,142535,23643.54208,7152274,64,64.26,61.55,106.40,NaN
2,2018-2019,Nov,2,ANDHRA PRADESH,211,Y.S.R,9831635,206.466248,59,4022,...,567546,1164238,137615,27883.60743,8591158,68,70.41,73.29,107.61,NaN
3,2018-2019,Sep,2,ANDHRA PRADESH,201,SRIKAKULAM,17114336,197.734116,46,9108,...,577694,1309583,89937,33824.03204,11557724,60,77.73,77.34,103.85,NaN
4,2018-2019,Sep,2,ANDHRA PRADESH,202,VIZIANAGARAM,14897034,196.608709,46,10772,...,517720,1217417,120755,32315.21691,10553260,55,80.80,79.72,105.49,NaN


In [6]:
df = df.dropna(subset=['Average_Wage_rate_per_day_per_person'])

### Global Z-Score Method

In [7]:
from scipy.stats import zscore
import numpy as np

z = np.abs(zscore(df['Average_Wage_rate_per_day_per_person']))
df_z = df[z < 3]

In [8]:
print("Original rows:", df.shape[0])
print("After Z-score:", df_z.shape[0])

Original rows: 72576
After Z-score: 72560


In [9]:
df_z['Average_Wage_rate_per_day_per_person'].max()

np.float64(125792.056)

### State-Wise IQR Method

In [10]:
def remove_outliers(group):
    
    Q1 = group['Average_Wage_rate_per_day_per_person'].quantile(0.25)
    Q3 = group['Average_Wage_rate_per_day_per_person'].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return group[(group['Average_Wage_rate_per_day_per_person'] >= lower) &
                 (group['Average_Wage_rate_per_day_per_person'] <= upper)]

df_state_iqr = df.groupby('state_name', group_keys=False).apply(remove_outliers)

C:\Users\Shubhangi\AppData\Local\Temp\ipykernel_12772\3578942849.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_state_iqr = df.groupby('state_name', group_keys=False).apply(remove_outliers)


In [11]:
print("Original rows:", df.shape[0])
print("After State-Wise IQR:", df_state_iqr.shape[0])

Original rows: 72576
After State-Wise IQR: 61700


In [12]:
df_state_iqr['Average_Wage_rate_per_day_per_person'].max()

np.float64(370.2568368)